# A Shift-Reduce Parser for Arithmetic Expressions

In this notebook we implement a generic *shift reduce parser*.  The parse table that we use 
implements the following grammar for arithmetic expressions:
$$
  \begin{eqnarray*}
  \mathrm{expr}        & \rightarrow & \mathrm{expr}\;\;\texttt{'+'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{expr}\;\;\texttt{'-'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{product}                                    \\[0.2cm]
  \mathrm{product}     & \rightarrow & \mathrm{product}\;\;\texttt{'*'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{product}\;\;\texttt{'/'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{factor}                                     \\[0.2cm]
  \mathrm{factor}      & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}     \\
                       & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

This notebook extends the notebook `01-Shift-Reduce-Parser-Pure.ipynb`:  The parser prints every step of its
computation and it builds a parse tree, which is then displayed as a graph.

## Implementing a Scanner

In order to parse, we need a scanner.  We use the scanner from the notebook `01-Shift-Reduce-Parser-Pure.ipynb`
with one change:  As we want to build a parse tree, the scanner has to keep the values of the tokens.  For example,
the parse tree of the string `'1 + 2'` should contain the numbers `1` and `2`, not just the token `NUMBER` twice.

In [ ]:
import re

The function `tokenize` scans the string `s` into a list of tokens using Python's regular expressions.
Every token is a pair of the form `(type, value)`:
* The *type* is the name of the token.  The parser uses the type to look up the action table.
* The *value* is the part of `s` that has been recognized.  The parser stores the value in the parse tree.

The scanner distinguishes between
* whitespace, which is discarded,
* numbers, which have the type `'NUMBER'` and their digits as value,
* arithmetical operators and parentheses, which serve both as type and as value,
* all remaining characters, which are treated as lexical errors and get the type `'ERROR'`.
  As the action table has no entries for this type, the parser reports a syntax error when it reaches such a token.

See below for an example.

In [ ]:
def tokenize(s: str) -> list[tuple[str, str]]:
    '''Transform the string s into a list of (type, value) tuples.'''
    lexSpec = r'''([ \t\n]+)      |  # blanks and tabs
                  ([1-9][0-9]*|0) |  # number
                  ([-+*/()])      |  # arithmetical operators
                  (.)                # unrecognized character
               '''
    tokenList = re.findall(lexSpec, s, re.VERBOSE)
    result    = []
    for ws, number, operator, error in tokenList:
        if ws:        
            continue
        elif number:
            # Store the generic type 'NUMBER' and the actual value string
            result.append(('NUMBER', number))
        elif operator:
            # For operators, type and value are identical
            result.append((operator, operator))
        else:
            result.append(('ERROR', error))
    return result

In [ ]:
tokenize('1 + 2 * (3 - 4)')

## Implementing the Shift-Reduce Parser

Next, we define some type aliases.  Compared with the notebook `01-Shift-Reduce-Parser-Pure.ipynb`, there are
three new types:
- `MarkedRule` is the type of *marked rules*.  A marked rule is a grammar rule with a dot `•` somewhere in its body.
  It is stored as a string, e.g. `'e → e • "+" p'`.
- `StateTable` is the type of a dictionary that maps every state to the list of its marked rules.  The marked rules
  are listed in the order in which they have been added to the state.
- `ParseTree` is the type of parse trees.  A leaf is the value of a token, e.g. `'1'` or `'+'`.  An inner node is
  a pair of the form `(head, children)`, where `head` is a variable and `children` is the tuple of the subtrees of
  this node.

In [ ]:
Token       = str
Variable    = str
Symbol      = Token | Variable
State       = str
MarkedRule  = str
Rule        = tuple[Variable, tuple[Symbol, ...]]
Action      = str | tuple[str, State] | tuple[str, Rule]
ActionTable = dict[tuple[State, Token], Action]
GotoTable   = dict[tuple[State, Variable], State]
StateTable  = dict[State, list[MarkedRule]]
type ParseTree = str | tuple[Variable, tuple[ParseTree, ...]]

Assume a grammar $G = \langle V, T, R, s \rangle$ is given.  A  *shift-reduce parser*
is defined as a 4-Tuple
$$P = \langle Q, q_0, \texttt{action}, \texttt{goto} \rangle$$
where
- $Q$ is the set of *states* of the shift-reduce parser.  

  For the purpose of the shift-reduce-parser, states are purely abstract. 
- $q_0 \in Q$ is the *start state*.
- $\texttt{action}$ is a function taking two arguments. The first argument is a state $q \in Q$
  and the second argument is a token $t \in T$.  The result of this function is an element from the set
  $$\texttt{Action} :=
       \bigl\{ \langle\texttt{shift}, q\rangle  \mid q \in Q \bigr\}               \cup 
       \bigl\{ \langle\texttt{reduce}, r\rangle \mid r \in R \bigr\} \cup 
       \bigl\{ \texttt{accept},\, \texttt{error}  \bigr\}.                         
  $$
  Here `shift`, `reduce`, `accept`, and `error` are strings that serve to
  distinguish the different kinds of results returned by the function 
  `action`:
  $$\texttt{action}: Q \times T \rightarrow \texttt{Action}.$$
- `goto` is a function that takes a state $q \in Q$ and a syntactical variable
  $v \in V$ and computes a new state:
  $$\texttt{goto}: Q \times V \rightarrow Q.$$

The class `ShiftReduceParser` maintains three tables that are implemented as dictionaries:
- `mActionTable` encodes the function $\texttt{action}: Q \times T \rightarrow \texttt{Action}$.
- `mGotoTable` encodes the function $\texttt{goto}: Q \times V \rightarrow Q$.
- `mStateTable` maps every state to the list of its marked rules.  This table is not needed for parsing.  The parser only uses it to print the states.

The constructor takes these tables as arguments and stores them in the member variables `mActionTable`,
`mGotoTable`, and `mStateTable`.

In [ ]:
from typing import Literal

class ShiftReduceParser():
    def __init__(self, actionTable: ActionTable, gotoTable: GotoTable, stateTable: StateTable) -> None:
        self.mActionTable: ActionTable = actionTable
        self.mGotoTable  : GotoTable   = gotoTable
        self.mStateTable : StateTable  = stateTable
        
    def parse(self, TL: list[tuple[str, str]]) -> ParseTree | Literal[False]:
        ... 

The method `parse` takes a list of tokens `TL` as its argument, where every token is a pair of the form
`(type, value)`.  If the token list can be parsed successfully, the method returns the parse tree of `TL`,
otherwise it returns `False`.  The algorithm that is applied is known as *shift/reduce parsing*.  It is the same
algorithm as in the notebook `01-Shift-Reduce-Parser-Pure.ipynb`, but it has two extensions:
1. Before every step, the method prints
   - the stack of states,
   - the stack of symbols, followed by the symbol `|` and the values of the remaining tokens,
   - the marked rules of the state on top of the stack of states, and
   - the action that is taken.

   The marked rules are printed in the order in which they have been added to the state.
2. The method maintains a third stack, the list `Forest`, which contains the parse trees of the symbols on the
   symbol stack:
   - When a token is *shifted*, its value is pushed onto `Forest`.
   - When the parser *reduces* with a rule $A \rightarrow X_1 \cdots X_n$, the parse trees of
     $X_1$, $\cdots$, $X_n$ are the top $n$ elements of `Forest`.  They are removed from `Forest` and replaced
     by the parse tree `(A, (t1, ..., tn))`, where `t1`, $\cdots$, `tn` are the parse trees of
     $X_1$, $\cdots$, $X_n$.
   - When the parser *accepts*, `Forest` contains exactly one parse tree.  This is the parse tree of
     the whole token list and it is returned.

In [ ]:
def parse(self, TL: list[tuple[str, str]]) -> ParseTree | Literal[False]:
    index                 = 0      # points to next token
    Symbols: list[Symbol] = []     # stack of grammar symbols (stores types, e.g., 'NUMBER')
    States : list[State]  = ['s0'] # stack of states
    # The Forest stack stores the tree nodes/values
    Forest : list[ParseTree] = []
    TL = TL + [('$', '$')]
    while True:
        q = States[-1]
        # Unpack the current lookahead token into type and value
        token_type, token_value = TL[index]
        # Debug output (optional formatting)
        # We map the remaining tokens to their string representation for cleaner printing
        remaining_tokens = [val for _, val in TL[index:]]
        print(f'States:  [ {", ".join(States)} ]')
        print('Symbols:', " ".join(Symbols + ['|'] + remaining_tokens).strip())
        # Print the marked rules of the state on top of the state stack in the order in which they were added.
        print(f'State:   {q} = {{ {", ".join(self.mStateTable[q])} }}')
        # Use token_type for the table lookup
        action = self.mActionTable.get((q, token_type), 'error')
        match action:
            case 'error': 
                print(f'Action({q}, {token_type}) undefined.')
                print('Syntax error!\n')
                return False
            case 'accept':
                print('Accepting!\n')
                return Forest[0] 
            case 'shift', s:
                print(f'Shifting state {s}\n')
                Symbols += [token_type] # Stack tracks the grammar type
                States  += [s]
                # Forest tracks the actual value ('1', '+', etc.)
                Forest += [token_value]
                index   += 1
            case 'reduce', rule:
                head, body = rule
                print(f'Reducing with rule {head} → {" ".join(body)}')
                n        = len(body)
                children = tuple(Forest[len(Forest)-n:])
                Forest   = Forest[:len(Forest)-n]
                Forest  += [(head, children)]
                Symbols = Symbols[:len(Symbols)-n]
                States  = States [:len(States)-n]
                Symbols = Symbols + [head]
                state   = States[-1]
                States += [ self.mGotoTable[state, head] ]
            
ShiftReduceParser.parse = parse 
del parse

The notebook `03-Parse-Table.ipynb` contains the parse table for the grammar of arithmetic expressions
that is given above.  Besides the action table `actionTable` and the goto table `gotoTable`, it defines the
state table `stateTable`.

In [ ]:
%run 03-Parse-Table.ipynb

**Note:** The type checker *basedpyright* analyzes every notebook on its own.  Since it cannot follow the
magic command `%run`, the next cell declares the names that are defined in the notebook `03-Parse-Table.ipynb`.
These declarations are never executed, since `TYPE_CHECKING` is `False` at run time.  The script
`check_declarations.py` in the directory `Python` checks that they agree with the definitions.

In [ ]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:  # declarations for the type checker, never executed
    actionTable: dict[tuple[State, Token], Action]
    gotoTable:   dict[tuple[State, Variable], State]
    stateTable:  dict[State, list[MarkedRule]]

## Visualization

We use the module `graphviz` to display parse trees as graphs.

In [ ]:
import graphviz

The function `draw_tree` takes a parse tree `tree` and returns a `graphviz.Digraph` that displays this tree.
Inner nodes are labelled with the head of the corresponding grammar rule and are drawn as ovals, while leaves are
labelled with the values of the tokens and are drawn as boxes.  As different nodes can have the same label,
every node gets a unique name of the form `node`$k$, which is generated by the function `get_id`.
The nested function `visit` adds the parse tree `node` to the graph and connects its root with the node named
`parent_id`.  Then it visits the children of `node` recursively.

In [ ]:
def draw_tree(tree: ParseTree) -> graphviz.Digraph:
    dot = graphviz.Digraph(format='png')
    # Counter to generate unique IDs for each node in the graph
    counter = 0

    def get_id() -> str:
        nonlocal counter
        counter += 1
        return f'node{counter}'

    def visit(node: ParseTree, parent_id: str | None = None) -> None:
        # Create a unique ID for the current node
        node_id = get_id() 
        if isinstance(node, tuple):
            # It's a Non-Terminal: (Head, (Children...))
            head, children = node
            # Draw the Non-Terminal node (default shape is oval)
            dot.node(node_id, label=head)
            # Connect to parent if it exists
            if parent_id:
                dot.edge(parent_id, node_id)
            # Recursively visit all children
            for child in children:
                visit(child, node_id)
        else:
            # It's a Terminal: string value (e.g., '1', '+')
            label = str(node)
            # Draw the Terminal node with a distinct shape/font
            dot.node(node_id, label=label, shape='box', fontname='Courier')
            if parent_id:
                dot.edge(parent_id, node_id)
                
    # Start the traversal from the root
    visit(tree)
    
    return dot

## Testing

Given a string `s`, the function `test` transforms `s` into a list of tokens, prints this list, and parses it.
If the parse is successful, the function returns the parse tree as a graph.  Hence, the parse tree is displayed
if `test` is called in the last line of a cell.  Otherwise, the function returns `None`.

In [ ]:
def test(s: str) -> graphviz.Digraph | None:
    parser = ShiftReduceParser(actionTable, gotoTable, stateTable)
    TL     = tokenize(s)
    print(f'tokenlist: {TL}\n')
    tree = parser.parse(TL)
    if tree:
        print('Parse successful!')
        return draw_tree(tree)
    else:
        print('Parse failed!')

In [ ]:
pt = test('1 + 2 * 3')
pt

In [ ]:
test('1 + 2 * (3 - 4)')

In [ ]:
test('1 + * 2')